# 05 · Modelo final: dispersão por região, dependência espacial e extensões rejeitadas

Ponto de partida: `bb_selecao`, o beta-binomial escolhido no notebook 02 (seleção de variáveis por BIC,
fusão de Nordeste e Norte na média) e validado pelo notebook 03. Cada extensão é mantida só se melhorar
seu diagnóstico **e** o erro médio por UF no LOSO ficar no limite de aceitação. As regras estão escritas
antes dos resultados, em cada Parte.

- Parte 13: dispersão do beta-binomial por região (decisão pelo Wald robusto).
- Parte 14: dependência espacial, ESF e SLX, com a aceitação pela validação cruzada.
- Parte 15: erros-padrão de Conley do modelo final e robustez do corte em 2022.
- Parte 16: estimativas regionais e por UF do modelo final (LOSO).
- Parte 17: extensões rejeitadas, krigagem e CAR/BYM dos resíduos.

Tabelas em `results/tables/final_*`, figuras em `results/figures/final_*.html`.


In [1]:
import sys, json, warnings, time
from pathlib import Path
from dataclasses import asdict
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import get_context
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "notebooks"))
from src import especificacoes as E, modelos as M, espacial as S
import graficos_diagnostico as G
import geopandas as gpd
pd.options.display.float_format = "{:,.4f}".format
T0 = time.time()
RECALCULAR = False   # True: refaz a validação cruzada das extensões
TAB, FIG, CV = ROOT / "results/tables", ROOT / "results/figures", ROOT / "results/cv"
for p in (TAB, FIG, CV): p.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(ROOT / "data/processed/municipios.parquet").reset_index(drop=True)
mapa = E.MAPA_2026
ids = df["id_municipio"].astype(str)
geo = pd.read_parquet(ROOT / "data/processed/municipios_geo.parquet")
geo["id_municipio"] = geo["id_municipio"].astype(str)
geo = geo.set_index("id_municipio").loc[ids].reset_index()
gdf = gpd.GeoDataFrame(geo, geometry=gpd.GeoSeries.from_wkb(geo["geometry"]), crs="EPSG:4674")
ARQ_EV = ROOT / "data/processed/autovetores_esf.parquet"
if ARQ_EV.exists():
    ev = pd.read_parquet(ARQ_EV); ev["id_municipio"] = ev["id_municipio"].astype(str)
else:
    ev_df, ev_I = S.autovetores_esf(gdf, k=100)
    ev = ev_df.reset_index(drop=True).assign(id_municipio=ids.to_numpy())
    ev[["id_municipio"] + list(ev_df.columns)].to_parquet(ARQ_EV)
    pd.DataFrame({"autovetor": ev_df.columns, "I_moran": np.asarray(ev_I)}).to_csv(TAB / "final_esf_moran_autovetores.csv", index=False)
ev = ev.set_index("id_municipio").loc[ids].reset_index(drop=True)
df = pd.concat([df, ev], axis=1)
sel = json.loads((TAB / "selecao_variaveis.json").read_text()); E.registrar_selecao(sel["variaveis"], tuple(tuple(g) for g in sel["regiao_fundir"]))
E.definir_dados_cv(df, mapa)
y, n = df[mapa["y"]].to_numpy(float), df[mapa["n"]].to_numpy(float)
uf, reg = df[mapa["uf"]].astype(str).to_numpy(), df[mapa["regiao"]].astype(str).to_numpy()
UFS_ = sorted(set(uf)); REGS = sorted(set(reg))
W = S.pesos_queen(gdf)
BASE = E.CANDIDATOS["bb_selecao"]
print(len(df), "municípios;", ev.shape[1], "autovetores;", "termos de bb_selecao:", len(BASE.termos.vars), "contínuas")

5570 municípios; 100 autovetores; termos de bb_selecao: 12 contínuas


## Parte 13 · Dispersão do beta-binomial por região

**Ponto de partida.** A média de `bb_selecao` tem 12 termos e funde Nordeste e Norte numa só dummy de região
(regra da Parte 8 do notebook 02). A dispersão (log θ = Zγ) é um submodelo à parte: usa os cinco níveis de
região (Sudeste como referência), sem a fusão da média.

**Regra (escrita antes do resultado).** Opções de dispersão z ∈ {`const`, `regiao`, `regiao_logn`}, ajustadas a todos
os dados. O teste da razão de verossimilhanças (LRT) e o BIC são reportados só como **motivação**: supõem
municípios independentes e, com resíduos espacialmente correlacionados, rejeitam demais. A **decisão** vem de um
Wald sequencial em γ com a covariância de Conley do próprio ajuste (corte = regra da Parte 15 aplicada ao
correlograma dos resíduos quantílicos desse ajuste, mesmas faixas; resíduos aleatorizados, semente 0).

- Passo A: as 4 dummies de região no ajuste `regiao` (contra `const`).
- Passo B: `log_n` (1 gl) no ajuste `regiao_logn`; só é testado se A for mantido.
- Um passo é mantido se p < 0,01 sob Conley ×1 (qui-quadrado). Conley ×0,5, ×2 e conglomerado por UF são
  reportados como F com 26 gl no denominador (27 UFs − 1).
- Resultado: `const` se A falha; `regiao` se B falha; senão `regiao_logn`. A adoção final ainda passa pela aceitação
  por validação cruzada da Parte 14.


In [2]:
OPCOES = ["const", "regiao", "regiao_logn"]
AJ14, PI14, TH14, R14 = {}, {}, {}, {}
for o in OPCOES:
    if o == "const":
        c = BASE
    else:
        c = E.registrar_extensao(f"bb_disp_{o}", disp=o)
    a = c.ajustar(df, mapa); AJ14[o] = a
    PI14[o] = np.asarray(c.prever(a, df, mapa), float)
    TH14[o] = E.prever_theta(a, df, mapa)
    R14[o] = G.residuos_quantilicos_rapido(y, n, PI14[o], theta=TH14[o], rng=0)
nobs = len(df)
L = []
for o in OPCOES:
    a = AJ14[o]; ll, k = a.loglik, a.k
    st = max(0.0, 2 * (ll - AJ14["const"].loglik)); gl = k - AJ14["const"].k
    L.append({"opcao": o, "llf": ll, "k": k, "AIC": 2 * k - 2 * ll, "BIC": k * np.log(nobs) - 2 * ll,
              "LRT_stat": st if gl else np.nan, "LRT_gl": gl if gl else np.nan,
              "LRT_p": stats.chi2.sf(st, gl) if gl else np.nan})
DISP_T = pd.DataFrame(L); DISP_T.to_csv(TAB / "final_dispersao.csv", index=False)
DP14 = pd.DataFrame({o: pd.Series(R14[o]).groupby(reg).std(ddof=1) for o in OPCOES}).T
DP14["max_abs_dp_menos_1"] = (DP14[REGS] - 1).abs().max(axis=1)
DP14.rename_axis("opcao").to_csv(TAB / "final_dp_regiao.csv")
print(DISP_T.to_string(index=False)); print("\nDP do resíduo quantílico por região"); print(DP14.to_string())

      opcao          llf  k         AIC         BIC  LRT_stat  LRT_gl  LRT_p
      const -36,340.4979 19 72,718.9958 72,844.8737       NaN     NaN    NaN
     regiao -36,163.2899 23 72,372.5798 72,524.9582  354.4161  4.0000 0.0000
regiao_logn -36,087.5305 24 72,223.0611 72,382.0647  505.9348  5.0000 0.0000

DP do resíduo quantílico por região
             Centro-Oeste  Nordeste  Norte  Sudeste    Sul  max_abs_dp_menos_1
const              0.9505    1.1828 1.4342   0.8039 0.9246              0.4342
regiao             1.0049    1.0336 1.0719   1.0231 1.0233              0.0719
regiao_logn        0.9973    1.0268 1.0872   1.0097 1.0096              0.0872


In [3]:
ESCOLHA14 = DISP_T.sort_values("BIC").iloc[0]["opcao"]
lin = DISP_T.set_index("opcao").loc[ESCOLHA14]
if ESCOLHA14 == "const":
    DISP_OK, MOTIVO14 = False, "const tem o menor BIC; nenhuma dispersão variável escolhida"
else:
    p_ok = bool(lin.LRT_p < 0.01); d_ok = bool(DP14.loc[ESCOLHA14, "max_abs_dp_menos_1"] < DP14.loc["const", "max_abs_dp_menos_1"])
    DISP_OK = p_ok and d_ok
    MOTIVO14 = (f"LRT p={lin.LRT_p:.3g} ({'<' if p_ok else '>='} 0,01); max|DP-1| {DP14.loc['const','max_abs_dp_menos_1']:.3f} (const) -> "
                f"{DP14.loc[ESCOLHA14,'max_abs_dp_menos_1']:.3f} ({ESCOLHA14}) ({'melhora' if d_ok else 'não melhora'})")
DISP = ESCOLHA14 if DISP_OK else "const"
print("[motivação, não decide] escolhida por BIC:", ESCOLHA14, "| aceita:", DISP_OK, "|", MOTIVO14, "| dispersão adotada:", DISP)
fig = go.Figure([go.Bar(name=o, x=REGS, y=DP14.loc[o, REGS].values) for o in OPCOES])
fig.add_hline(y=1, line_dash="dash", line_color="#888")
fig.update_layout(barmode="group", title="DP do resíduo quantílico por região, por opção de dispersão (ideal = 1)", yaxis_title="DP", template="plotly_white", height=420)
G.salvar(fig, FIG / "final_dp_regiao.html"); fig.show()

[motivação, não decide] escolhida por BIC: regiao_logn | aceita: True | LRT p=4.18e-107 (< 0,01); max|DP-1| 0.434 (const) -> 0.087 (regiao_logn) (melhora) | dispersão adotada: regiao_logn


**Leitura (motivação).** O BIC escolhe `regiao_logn` (72.382,1 contra 72.844,9 de `const`), e o LRT rejeita a dispersão constante (χ² = 505,9 com 5 gl; p = 4,2e-107). O desvio residual por região fica perto de 1: o máximo |DP − 1| cai de 0,434 (`const`; Norte com DP 1,43) para 0,087 (`regiao_logn`). O LRT supõe municípios independentes, então não decide: a decisão é do Wald robusto abaixo.

*Números da execução de 2026-11-18; se o notebook for reexecutado, confira as saídas acima.*

In [4]:
FAIXAS = [0, 10, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000]
lat, lon = df["lat"].to_numpy(float), df["lon"].to_numpy(float)
def corte_de(cg):
    ab = cg[cg.correlacao < 0.05]
    return float(ab.iloc[0]["ini_km"]) if len(ab) else float(cg.iloc[-1]["fim_km"])
def passo_wald(passo, o, sel):
    a = AJ14[o]; mod = a.modelo; nm = list(a.nomes)
    ind = [i for i, s in enumerate(nm) if sel(s)]; q = len(ind)
    corte = corte_de(S.correlograma(R14[o], lat, lon, FAIXAS))
    covs = {"conley_05": S.cov_conley(mod, lat, lon, corte * 0.5), "conley_1": S.cov_conley(mod, lat, lon, corte),
            "conley_2": S.cov_conley(mod, lat, lon, corte * 2), "cluster_uf": S.cov_cluster(mod, uf, correcao=True)}
    linhas = []
    for tag, cv in covs.items():
        for tipo, gd in ([("chi2", None)] if tag == "conley_1" else []) + [("F", len(UFS_) - 1)]:
            est, gl, p = S.wald_robusto(mod.params, cv, ind, gl_den=gd)
            linhas.append(dict(passo=passo, ajuste=o, termos=";".join(nm[i] for i in ind), covariancia=tag, corte_km=corte * (0.5 if tag == "conley_05" else 2 if tag == "conley_2" else 1) if tag != "cluster_uf" else np.nan,
                               teste=tipo, estat=est, gl=gl, gl_den=gd if gd else np.nan, p=p))
    return pd.DataFrame(linhas), corte
WA, corte_A = passo_wald("A", "regiao", lambda s: s.startswith("disp_reg_"))
A_OK = bool(WA[(WA.covariancia == "conley_1") & (WA.teste == "chi2")].p.iloc[0] < 0.01)
if A_OK:
    WB, corte_B = passo_wald("B", "regiao_logn", lambda s: s == "disp_log_n")
    B_OK = bool(WB[(WB.covariancia == "conley_1") & (WB.teste == "chi2")].p.iloc[0] < 0.01)
    WALD_T = pd.concat([WA, WB], ignore_index=True)
else:
    B_OK, corte_B, WALD_T = False, np.nan, WA
WALD_T["mantido"] = WALD_T.passo.map({"A": A_OK, "B": B_OK})
WALD_T.to_csv(TAB / "final_dispersao_wald.csv", index=False)
print(WALD_T.to_string(index=False))
DISP_ANT, DISP_OK_ANT = DISP, DISP_OK
DISP = "regiao_logn" if (A_OK and B_OK) else "regiao" if A_OK else "const"
DISP_OK = DISP != "const"
pa = WA[(WA.covariancia == "conley_1") & (WA.teste == "chi2")].p.iloc[0]
MOTIVO14 = (f"Wald robusto (Conley x1, chi2): passo A (dummies de região) p={pa:.3g} -> {'mantido' if A_OK else 'descartado'}"
            + (f"; passo B (log_n) p={WB[(WB.covariancia == 'conley_1') & (WB.teste == 'chi2')].p.iloc[0]:.3g} -> {'mantido' if B_OK else 'descartado'}" if A_OK else "")
            + f". Antes (LRT/BIC): {DISP_ANT}")
WALD_DEC = {"dispersao_adotada": DISP, "A_mantido": A_OK, "B_mantido": B_OK, "corte_A_km": corte_A, "corte_B_km": corte_B,
            "tabela": WALD_T.to_dict(orient="records"), "motivo": MOTIVO14}
print("dispersão adotada:", DISP, "|", MOTIVO14)

passo      ajuste                                                              termos covariancia  corte_km teste   estat  gl  gl_den      p  mantido
    A      regiao disp_reg_Centro-Oeste;disp_reg_Nordeste;disp_reg_Norte;disp_reg_Sul   conley_05  100.0000     F 22.8341   4 26.0000 0.0000     True
    A      regiao disp_reg_Centro-Oeste;disp_reg_Nordeste;disp_reg_Norte;disp_reg_Sul    conley_1  200.0000  chi2 57.3710   4     NaN 0.0000     True
    A      regiao disp_reg_Centro-Oeste;disp_reg_Nordeste;disp_reg_Norte;disp_reg_Sul    conley_1  200.0000     F 14.3428   4 26.0000 0.0000     True
    A      regiao disp_reg_Centro-Oeste;disp_reg_Nordeste;disp_reg_Norte;disp_reg_Sul    conley_2  400.0000     F 14.9819   4 26.0000 0.0000     True
    A      regiao disp_reg_Centro-Oeste;disp_reg_Nordeste;disp_reg_Norte;disp_reg_Sul  cluster_uf       NaN     F  9.2556   4 26.0000 0.0001     True
    B regiao_logn                                                          disp_log_n   conley_05  1

**Leitura.** Os dois passos se mantêm em todas as covariâncias. Passo A (dummies de região na dispersão): χ² = 57,4, p = 1,0e-11 com Conley a 200 km; o caso mais fraco é o conglomerado por UF (F = 9,3, p = 1e-4). Passo B (log n): χ² = 64,8, p = 8,3e-16; o caso mais fraco é o conglomerado por UF (F = 40,1). A dependência espacial infla os EPs, mas não a ponto de apagar a dispersão por região e porte: a dispersão de `bb_selecao` passa de `const` para `regiao_logn` (cinco níveis de região, Nordeste e Norte separados na dispersão).

*Números da execução de 2026-11-18; se o notebook for reexecutado, confira as saídas acima.*

## Parte 14 · Dependência espacial: ESF e SLX

Dois remédios para a correlação espacial dos resíduos são testados, cada um com o diagnóstico próprio e depois
com a aceitação pela validação cruzada (fim da Parte).

### Filtro espacial de Moran (ESF)

**Regra (escrita antes do resultado).** Entram em X os k primeiros autovetores de Moran (ordenados
por I decrescente; dependem só da geometria, sem vazamento de y), k ∈ {0, 10, 25, 50, 100}, com a
dispersão adotada na Parte 13 se foi aceita (senão `const`). Escolha do k pelo menor BIC.
O filtro passa no diagnóstico se o k escolhido for > 0 e o I de Moran dos resíduos quantílicos (pesos Queen,
semente 0, 999 permutações) cair em relação a k = 0. Tabela por k: llf, nº de parâmetros, BIC, I, p_sim.


In [5]:
KS = [0, 10, 25, 50, 100]
AJ15, R15, L = {}, {}, []
for k in KS:
    c = E.registrar_extensao(f"bb_esf_k{k}", disp=DISP, esf=k)
    a = c.ajustar(df, mapa); AJ15[k] = a
    pi = np.asarray(c.prever(a, df, mapa), float)
    R15[k] = G.residuos_quantilicos_rapido(y, n, pi, theta=E.prever_theta(a, df, mapa), rng=0)
    m = S.moran_residuos(R15[k], W, permutacoes=999, semente=0)
    L.append({"k_esf": k, "dispersao": DISP, "llf": a.loglik, "n_params": a.k, "BIC": a.k * np.log(nobs) - 2 * a.loglik,
              "I_moran": m["I"], "p_sim": m["p_sim"], "dp_residuo": R15[k].std(ddof=1)})
ESF_T = pd.DataFrame(L); ESF_T.to_csv(TAB / "final_esf_k.csv", index=False)
print(ESF_T.to_string(index=False))

 k_esf   dispersao          llf  n_params         BIC  I_moran  p_sim  dp_residuo
     0 regiao_logn -36,087.5305        24 72,382.0647   0.3424 0.0010      1.0204
    10 regiao_logn -36,011.1250        34 72,315.5051   0.3192 0.0010      1.0213
    25 regiao_logn -35,820.7037        49 72,064.0397   0.2730 0.0010      1.0245
    50 regiao_logn -35,651.5219        74 71,941.3049   0.2296 0.0010      1.0295
   100 regiao_logn -35,530.2807       124 72,130.0799   0.1969 0.0010      1.0284


In [6]:
K_ESC = int(ESF_T.sort_values("BIC").iloc[0]["k_esf"])
I0, Ik = float(ESF_T.set_index("k_esf").loc[0, "I_moran"]), float(ESF_T.set_index("k_esf").loc[K_ESC, "I_moran"])
ESF_OK = K_ESC > 0 and Ik < I0
MOTIVO15 = f"k escolhido por BIC = {K_ESC}; I de Moran {I0:.4f} (k=0) -> {Ik:.4f} (k={K_ESC})" + ("" if K_ESC > 0 else "; k=0 vence, sem filtro")
print("ESF aceito (diagnóstico):", ESF_OK, "|", MOTIVO15)
fig = go.Figure(go.Scatter(x=ESF_T.k_esf, y=ESF_T.I_moran, mode="lines+markers", marker=dict(size=9, color="#1b6ca8")))
fig.update_layout(title="I de Moran dos resíduos quantílicos contra o nº de autovetores (k)", xaxis_title="k", yaxis_title="I de Moran", template="plotly_white", height=400)
G.salvar(fig, FIG / "final_moran_k.html"); fig.show()

ESF aceito (diagnóstico): True | k escolhido por BIC = 50; I de Moran 0.3424 (k=0) -> 0.2296 (k=50)


**Leitura.** Como diagnóstico, o ESF funciona: com k = 50 autovetores (escolha do BIC: 71.941,3 contra 72.382,1 em k = 0) o I de Moran do resíduo cai de 0,342 para 0,230. A aceitação final depende da validação cruzada por UF, abaixo: os autovetores são da malha inteira, e a UF retirada continua no filtro.

*Números da execução de 2026-11-18; se o notebook for reexecutado, confira as saídas acima.*

### Covariáveis defasadas espacialmente (SLX)

**Regra (escrita antes do resultado).** Como segundo remédio para o I de Moran residual do modelo base, tenta-se SLX:
W·x (vizinhança rainha, linhas padronizadas) entra em X. Com `disp` = dispersão adotada e `esf` = 0, comparam-se três
opções: **nenhuma**, **W·lag**, **W·(as 12 variáveis do modelo base)**. Escolha por BIC; reporta-se loglik, nº de
parâmetros, BIC e I de Moran dos resíduos quantílicos (mesma conta do ESF).

`SLX_OK` = a opção escolhida não é "nenhuma" **e** o I de Moran cai contra "nenhuma". A defasagem usa só
covariáveis, conhecidas em todo lugar: é calculada uma vez nos dados completos; a padronização é ajustada só na
dobra de treino. A partir daqui `df`/`mapa` são as versões com as colunas `w_*`.


In [7]:
df, mapa = S.adicionar_slx(df, mapa, list(BASE.termos.vars), W)
E.definir_dados_cv(df, mapa)
OPC_SLX = {"nenhuma": (), "W_lag": ("lag",), "W_12vars": tuple(BASE.termos.vars)}
AJ_SLX, L = {}, []
for o, ch in OPC_SLX.items():
    c = E.registrar_extensao(f"bb_slx_{o}", disp=DISP, esf=0, slx=ch)
    a = c.ajustar(df, mapa); AJ_SLX[o] = a
    pi = np.asarray(c.prever(a, df, mapa), float)
    r = G.residuos_quantilicos_rapido(y, n, pi, theta=E.prever_theta(a, df, mapa), rng=0)
    m = S.moran_residuos(r, W, permutacoes=999, semente=0)
    L.append({"opcao": o, "dispersao": DISP, "n_slx": len(ch), "llf": a.loglik, "n_params": a.k, "BIC": a.k * np.log(nobs) - 2 * a.loglik,
              "I_moran": m["I"], "p_sim": m["p_sim"], "dp_residuo": r.std(ddof=1)})
SLX_T = pd.DataFrame(L); SLX_T.to_csv(TAB / "final_slx.csv", index=False)
print(SLX_T.to_string(index=False))
SLX_ESC = SLX_T.sort_values("BIC").iloc[0]["opcao"]
I_none, I_esc = float(SLX_T.set_index("opcao").loc["nenhuma", "I_moran"]), float(SLX_T.set_index("opcao").loc[SLX_ESC, "I_moran"])
SLX_OK = SLX_ESC != "nenhuma" and I_esc < I_none
SLX_CHAVES = OPC_SLX[SLX_ESC] if SLX_OK else ()
MOTIVO_SLX = f"opção por BIC = {SLX_ESC}; I de Moran {I_none:.4f} (nenhuma) -> {I_esc:.4f} ({SLX_ESC})" + ("" if SLX_OK else "; SLX não aceito no diagnóstico")
SLX_DEC = {"opcao_bic": SLX_ESC, "bic": {r.opcao: float(r.BIC) for r in SLX_T.itertuples()}, "moran": {r.opcao: float(r.I_moran) for r in SLX_T.itertuples()},
           "diagnostico_ok": bool(SLX_OK), "chaves": list(SLX_CHAVES)}
print("SLX aceito (diagnóstico):", SLX_OK, "|", MOTIVO_SLX)

   opcao   dispersao  n_slx          llf  n_params         BIC  I_moran  p_sim  dp_residuo
 nenhuma regiao_logn      0 -36,087.5305        24 72,382.0647   0.3424 0.0010      1.0204
   W_lag regiao_logn      1 -36,079.8870        25 72,375.4027   0.3399 0.0010      1.0204
W_12vars regiao_logn     12 -35,932.4853        36 72,175.4761   0.3361 0.0010      1.0246
SLX aceito (diagnóstico): True | opção por BIC = W_12vars; I de Moran 0.3424 (nenhuma) -> 0.3361 (W_12vars)


**Leitura.** O BIC escolhe `W_12vars` (as 12 covariáveis defasadas), com BIC 72.175,5 contra 72.382,1 sem SLX (`W_lag`: 72.375,4). O ganho de ajuste é real, mas o I de Moran do resíduo quase não muda: 0,342 para 0,336. O SLX explica pouco da estrutura espacial; a correlação que resta não vem de covariáveis dos vizinhos.

*Números da execução de 2026-11-18; se o notebook for reexecutado, confira as saídas acima.*

### Aceitação pela validação cruzada (ESF e SLX)

**Regra (escrita antes do resultado).** Candidatos: `bb_disp` (dispersão escolhida, k = 0), `bb_esf` (`const`, k
escolhido), `bb_final` (as duas adições) e `bb_slx` (dispersão adotada, SLX escolhido). Só entram na CV as adições
cujo diagnóstico passou. Cada uma roda em 5-fold e LOSO. **Limite de aceitação** L = MAE por UF **aninhado** do
`bb_selecao` + `se_uf_pp` do melhor modelo no LOSO. O aninhado refaz a seleção de variáveis e a fusão de regiões dentro de cada dobra LOSO
(notebook 03, `cv_loso_aninhado.csv`); o melhor modelo e seu EP vêm de `cv_loso.csv`. Uma adição que falha o diagnóstico ou o limite é
descartada. O modelo final é `bb_selecao` mais as adições aceitas: se dispersão e ESF passam isoladamente, vale
`bb_final` só se ele também passar o limite (senão, a de menor erro por UF); se `bb_slx` for aceito, `modelo_final` =
`bb_slx`. `bb_slx` carrega a dispersão escolhida, então só é aceito se a dispersão também for aceita (quando o diagnóstico dela passou). Ressalva: as extensões ainda usam a seleção dos dados completos, de modo que o L aninhado é mais folgado do que uma comparação em pé de igualdade. Informativo: se o final está dentro da faixa de empate original (melhor + EP).


In [8]:
CAND = {}
if DISP_OK: CAND["bb_disp"] = dict(disp=DISP, esf=0)
if ESF_OK: CAND["bb_esf"] = dict(disp="const", esf=K_ESC)
if DISP_OK and ESF_OK: CAND["bb_final"] = dict(disp=DISP, esf=K_ESC)
if SLX_OK: CAND["bb_slx"] = dict(disp=DISP if DISP_OK else "const", esf=0, slx=tuple(SLX_CHAVES))
for nome, kw in CAND.items(): E.registrar_extensao(nome, **kw)   # antes do pool: os workers são forks
folds5, foldsL = M.folds_kfold(len(df), k=5, seed=0), M.folds_por_grupo(uf)
def _cv(nome, esq):
    arq = CV / f"{esq}_{nome}.parquet"
    return pd.read_parquet(arq)["pi"].to_numpy() if arq.exists() and not RECALCULAR else None
def rodar(nomes, esq, folds):
    pend = [k for k in nomes if _cv(k, esq) is None]
    saida = {}
    if pend:
        with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
            fut = {(k, i): ex.submit(E.prever_fold, k, tr, te) for k in pend for i, (tr, te) in enumerate(folds)}
            for k in pend:
                pi = np.full(len(df), np.nan)
                for i, (tr, te) in enumerate(folds): pi[te] = fut[(k, i)].result()
                pd.DataFrame({"pi": pi}).to_parquet(CV / f"{esq}_{k}.parquet"); saida[k] = pi
    for k in nomes:
        if k not in saida: saida[k] = _cv(k, esq)
    return saida
NOMES_CV = ["bb_selecao"] + list(CAND)
PI5, PIL = rodar(NOMES_CV, "kfold", folds5), rodar(NOMES_CV, "loso", foldsL)
print("NaN:", {k: int(np.isnan(v).sum()) for d in (PI5, PIL) for k, v in d.items() if np.isnan(v).any()} or "nenhum", "|", round(time.time() - T0), "s")
obs_uf = pd.Series(y).groupby(uf).sum() / pd.Series(n).groupby(uf).sum()
obs_reg = pd.Series(y).groupby(reg).sum() / pd.Series(n).groupby(reg).sum()
NH = E.n_loso(df)   # n̂: peso das previsões fora da amostra, igual ao notebook 03
def erros(pi, esq, folds):
    a_uf = (M.agregar(pi, NH, uf).drop("Brasil") - obs_uf).abs() * 100
    a_rg = (M.agregar(pi, NH, reg).drop("Brasil") - obs_reg).abs() * 100
    m = M.metricas(y, n, pi)
    por = [M.metricas(y[te], n[te], pi[te])["mae_pond_pp"] for _, te in folds]
    m.update(mae_uf_pp=a_uf.mean(), se_uf_pp=a_uf.std(ddof=1) / np.sqrt(len(a_uf)), mae_reg_pp=a_rg.mean(), max_reg_pp=a_rg.max(),
             se_mae_pond_pp=np.std(por, ddof=1) / np.sqrt(len(por)))
    return m
LIN = []
for esq, PI, folds in [("5-fold", PI5, folds5), ("LOSO", PIL, foldsL)]:
    for k in NOMES_CV: LIN.append({"esquema": esq, "modelo": k, **erros(PI[k], esq, folds)})
CVT = pd.DataFrame(LIN); CVT.to_csv(TAB / "final_cv.csv", index=False)
cols = ["esquema", "modelo", "mae_uf_pp", "se_uf_pp", "mae_pond_pp", "se_mae_pond_pp", "mae_reg_pp", "rmse_pond_pp", "deviance"]
print(CVT[[c for c in cols if c in CVT]].to_string(index=False))

NaN: nenhum | 164 s
esquema     modelo  mae_uf_pp  se_uf_pp  mae_pond_pp  se_mae_pond_pp  mae_reg_pp  rmse_pond_pp       deviance
 5-fold bb_selecao     0.8550    0.1481       1.1501          0.0306      0.2844        1.5309   132,822.7747
 5-fold    bb_disp     0.8630    0.1585       1.1390          0.0331      0.2610        1.5315   133,056.1140
 5-fold     bb_esf     0.6454    0.1432       1.1468          0.0478      0.3701        1.5230   129,597.3581
 5-fold   bb_final     0.6691    0.1543       1.1132          0.0433      0.3571        1.5047   126,886.2824
 5-fold     bb_slx     0.8459    0.1501       1.1483          0.0625      0.3406        1.5290   131,827.3526
   LOSO bb_selecao     1.0348    0.1558       1.3037          0.1147      0.3868        1.6933   161,254.8255
   LOSO    bb_disp     1.0345    0.1608       1.2708          0.1201      0.3174        1.6703   157,444.6331
   LOSO     bb_esf     1.4842    0.5599       4.7777          0.5630      1.9458       11.2664 8,893

In [9]:
loso = pd.read_csv(TAB / "cv_loso.csv").sort_values("mae_uf_pp").reset_index(drop=True)
melhor, se_melhor = float(loso.loc[0, "mae_uf_pp"]), float(loso.loc[0, "se_uf_pp"])
mae_bb_completo = float(loso.set_index("modelo").loc["bb_selecao", "mae_uf_pp"])
ANI = pd.read_csv(TAB / "cv_loso_aninhado.csv").iloc[0]   # saída do notebook 03: rode-o antes
assert ANI["modelo"] == "bb_selecao" and abs(float(ANI["mae_uf_pp_completo"]) - mae_bb_completo) < 1e-9, "cv_loso_aninhado.csv desatualizado"
mae_bb = float(ANI["mae_uf_pp_aninhado"])
LIMITE = mae_bb + se_melhor
LIMITE_SEM_ANINHAR = mae_bb_completo + se_melhor
FAIXA = melhor + se_melhor
Lm = CVT[CVT.esquema == "LOSO"].set_index("modelo")["mae_uf_pp"]
passa = {k: bool(Lm[k] <= LIMITE) for k in CAND}
ACEITA = {"disp": bool(DISP_OK and passa.get("bb_disp", False)), "esf": bool(ESF_OK and passa.get("bb_esf", False))}
if ACEITA["disp"] and ACEITA["esf"] and not passa.get("bb_final", False):
    pior = "esf" if Lm["bb_disp"] <= Lm["bb_esf"] else "disp"; ACEITA[pior] = False
FINAL = {(False, False): "bb_selecao", (True, False): "bb_disp", (False, True): "bb_esf", (True, True): "bb_final"}[(ACEITA["disp"], ACEITA["esf"])]
ACEITA["slx"] = bool(SLX_OK and passa.get("bb_slx", False) and (ACEITA["disp"] or not DISP_OK))
if ACEITA["slx"]: FINAL = "bb_slx"
cf = (E.CANDIDATOS.get(FINAL) or E.EXTENSOES[FINAL])
termos = {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(cf.termos).items()}
dec = {"disp": DISP, "k_esf": K_ESC,
       "aceitas": {"disp": ACEITA["disp"], "esf": ACEITA["esf"]},
       "motivos": {"disp": MOTIVO14 + f"; CV LOSO mae_uf={Lm.get('bb_disp', float('nan')):.4f} vs limite {LIMITE:.4f}",
                   "esf": MOTIVO15 + f"; CV LOSO mae_uf={Lm.get('bb_esf', float('nan')):.4f} vs limite {LIMITE:.4f}"},
       "diagnostico_ok": {"disp": bool(DISP_OK), "esf": bool(ESF_OK)},
       "mae_uf_loso": {k: float(v) for k, v in Lm.items()},
       "limite": float(LIMITE), "mae_uf_bb_selecao": mae_bb, "mae_uf_bb_selecao_sem_aninhar": mae_bb_completo,
       "limite_sem_aninhar": float(LIMITE_SEM_ANINHAR), "se_melhor_loso": se_melhor,
       "faixa_empate_original": float(FAIXA), "final_dentro_da_faixa_original": bool(Lm[FINAL] <= FAIXA),
       "modelo_final": FINAL, "termos_final": termos}
dec["aceitas"]["slx"] = ACEITA["slx"]; dec["diagnostico_ok"]["slx"] = bool(SLX_OK)
dec["motivos"]["slx"] = MOTIVO_SLX + f"; CV LOSO mae_uf={Lm.get('bb_slx', float('nan')):.4f} vs limite {LIMITE:.4f}"
dec["dispersao_wald"] = WALD_DEC; dec["slx"] = SLX_DEC
(TAB / "final_decisao.json").write_text(json.dumps(dec, indent=2, ensure_ascii=False, default=str))
print(json.dumps({k: dec[k] for k in ["disp", "k_esf", "aceitas", "motivos", "limite", "faixa_empate_original", "final_dentro_da_faixa_original", "modelo_final"]}, indent=2, ensure_ascii=False))
print("tempo total:", round(time.time() - T0), "s")

{
  "disp": "regiao_logn",
  "k_esf": 50,
  "aceitas": {
    "disp": true,
    "esf": false,
    "slx": true
  },
  "motivos": {
    "disp": "Wald robusto (Conley x1, chi2): passo A (dummies de região) p=1.03e-11 -> mantido; passo B (log_n) p=8.26e-16 -> mantido. Antes (LRT/BIC): regiao_logn; CV LOSO mae_uf=1.0345 vs limite 1.2515",
    "esf": "k escolhido por BIC = 50; I de Moran 0.3424 (k=0) -> 0.2296 (k=50); CV LOSO mae_uf=1.4842 vs limite 1.2515",
    "slx": "opção por BIC = W_12vars; I de Moran 0.3424 (nenhuma) -> 0.3361 (W_12vars); CV LOSO mae_uf=1.0777 vs limite 1.2515"
  },
  "limite": 1.2514921889195791,
  "faixa_empate_original": 1.1330912482398103,
  "final_dentro_da_faixa_original": true,
  "modelo_final": "bb_slx"
}
tempo total: 165 s


**Leitura.** O limite de aceitação é L = 1,1168 + 0,1347 = 1,2515 p.p. de MAE por UF: o MAE do `bb_selecao` com a seleção refeita em cada dobra (aninhado, notebook 03) mais o EP do melhor modelo no LOSO. O 1,0348 da tabela acima é o `bb_selecao` com a seleção feita nos dados completos (pondera as UFs pelo n̂ do eleitorado, como o notebook 03) e daria L = 1,1695; o aninhado é o honesto e o que vale, e fica no json como `mae_uf_bb_selecao` (o outro, como `mae_uf_bb_selecao_sem_aninhar`). No LOSO, `bb_disp` (1,035) e `bb_slx` (1,078) passam, com folga (L = 1,2515, antes 1,1695); `bb_esf` (1,484) e `bb_final` (1,454) não, com EP e erro ponderado muito maiores: o filtro espacial extrapola mal para uma UF nova, embora seja o melhor nas 5 dobras (0,645). Pela regra fixada antes, a dispersão por região e o SLX são aceitos e o modelo final é `bb_slx` (`bb_selecao` + `regiao_logn` + SLX das 12 variáveis; 36 parâmetros), dentro da faixa de empate original (1,133). As aceitações são as mesmas com L = 1,1695 ou 1,2515, e nenhum modelo fica entre os dois limites (a seleção das extensões segue com os dados completos, então o L aninhado é mais folgado do que seria numa comparação em pé de igualdade). Erro por UF do final: 0,846 (5-fold) e 1,078 (LOSO). O `bb_disp` tem MAE um pouco menor no LOSO, e a regra não exigia queda mínima do I de Moran; essa limitação está em [12-pendencias](../docs/pressupostos/12-pendencias.md).

*Números da execução de 2026-11-18; se o notebook for reexecutado, confira as saídas acima.*

## Parte 15 · Erros-padrão de Conley (HAC espacial) do modelo final

O modelo final (`modelo_final` de `final_decisao.json`) ainda deixa resíduos correlacionados no espaço
(Moran > 0). Os EPs baseados no modelo
supõem observações independentes, logo são pequenos demais. Esta Parte troca-os por EPs robustos.

**Regra (escrita antes de ver os EPs).**
1. Correlograma dos resíduos quantílicos (semente 0) do modelo final, faixas [ini, fim) em km:
   0, 10, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000.
2. **Corte de Conley = borda inferior da primeira faixa com correlação < 0,05.**
3. Sensibilidade: corte × 0,5 e × 2, e conglomerado por UF (27 grupos, com correção G/(G−1)·(N−1)/(N−k)),
   que é um limite conservador (trata cada UF como um bloco com correlação livre dentro dele).
4. Núcleo de Bartlett, distância de círculo máximo, sem correção de pequena amostra.
5. Um coeficiente "muda" se |z| > 1,96 com o EP do modelo e não com o EP de Conley (corte da regra), ou o inverso.

In [10]:
FIN = json.loads((TAB / "final_decisao.json").read_text())
NOME_F = FIN["modelo_final"]
CF = E.CANDIDATOS.get(NOME_F) or E.EXTENSOES[NOME_F]
AF = CF.ajustar(df, mapa)
NB_ = AF.extra["n_beta"]
PIF = np.asarray(CF.prever(AF, df, mapa), float); THF = E.prever_theta(AF, df, mapa)
RF = G.residuos_quantilicos_rapido(y, n, PIF, theta=THF, rng=0)
lat, lon = df["lat"].to_numpy(float), df["lon"].to_numpy(float)
FAIXAS = [0, 10, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000]
CG = S.correlograma(RF, lat, lon, FAIXAS); CG.to_csv(TAB / "final_correlograma.csv", index=False)
abaixo = CG[CG.correlacao < 0.05]
CORTE = float(abaixo.iloc[0]["ini_km"])
print("modelo final:", NOME_F, "| n_beta:", NB_, "| params:", len(AF.modelo.params), "| corte de Conley:", CORTE, "km")
print(CG.to_string(index=False))
fig = go.Figure(go.Bar(x=[f"{a:g}-{b:g}" for a, b in zip(CG.ini_km, CG.fim_km)], y=CG.correlacao, marker_color="#1b6ca8"))
fig.add_hline(y=0.05, line_dash="dash", line_color="#c2571a", annotation_text="limiar 0,05")
fig.update_layout(title=f"Correlograma dos resíduos quantílicos ({NOME_F}); corte = {CORTE:g} km", xaxis_title="faixa de distância (km)", yaxis_title="correlação média", template="plotly_white", height=420)
G.salvar(fig, FIG / "final_correlograma.html"); fig.show()

modelo final: bb_slx | n_beta: 30 | params: 36 | corte de Conley: 200.0 km
  ini_km     fim_km   pares  correlacao
  0.0000    10.0000     617      0.3764
 10.0000    25.0000   10314      0.3386
 25.0000    50.0000   35450      0.2342
 50.0000    75.0000   53238      0.1657
 75.0000   100.0000   68805      0.1237
100.0000   150.0000  176025      0.0911
150.0000   200.0000  217585      0.0619
200.0000   300.0000  531651      0.0300
300.0000   400.0000  609990      0.0050
400.0000   600.0000 1297109     -0.0248
600.0000   800.0000 1383337     -0.0095
800.0000 1,000.0000 1378747     -0.0150


### Robustez: estabilidade do corte em 2022

**Regra (escrita antes do resultado).** O corte de 2026 vem de uma só eleição. Ajusta-se a especificação final no 2º turno de 2022 (`MAPA_FIT_2022`, defasagem = 2018; com colunas `w_*` recalculadas nos dados de 2022 se o final tiver SLX) e aplica-se a mesma regra ao correlograma. Se 0,5·CORTE ≤ CORTE_2022 ≤ 2·CORTE, o CORTE de 2026 fica; senão CORTE = max(CORTE, CORTE_2022). Depois, a curva do EP contra o corte (50 a 800 km) para o modelo final em 2026. A Parte 15 (tabela de EPs) e a Parte 16 usam o CORTE final.

In [11]:
from src import backtest as BT
CORTE_2026 = CORTE
M22 = BT.MAPA_FIT_2022
df22, m22 = (S.adicionar_slx(df, M22, list(CF.termos.slx), W) if CF.termos.slx else (df, M22))
y22, n22 = df22[M22["y"]].to_numpy(float), df22[M22["n"]].to_numpy(float)
A22 = CF.ajustar(df22, m22)
PI22 = np.asarray(CF.prever(A22, df22, m22), float)
R22 = G.residuos_quantilicos_rapido(y22, n22, PI22, theta=E.prever_theta(A22, df22, m22), rng=0)
CG22 = S.correlograma(R22, lat, lon, FAIXAS); CG22.to_csv(TAB / "final_correlograma_2022.csv", index=False)
CORTE_2022 = corte_de(CG22)
if 0.5 * CORTE_2026 <= CORTE_2022 <= 2 * CORTE_2026:
    CORTE = CORTE_2026; MOTIVO_CORTE = f"CORTE_2022={CORTE_2022:g} km dentro de [{0.5*CORTE_2026:g}, {2*CORTE_2026:g}]; corte de 2026 mantido"
else:
    CORTE = max(CORTE_2026, CORTE_2022); MOTIVO_CORTE = f"CORTE_2022={CORTE_2022:g} km fora de [{0.5*CORTE_2026:g}, {2*CORTE_2026:g}]; usa o máximo"
print(CG22.to_string(index=False)); print(f"corte 2026 = {CORTE_2026:g} km | corte 2022 = {CORTE_2022:g} km | corte final = {CORTE:g} km |", MOTIVO_CORTE)
nomes_c = list(AF.nomes); mod_c = AF.modelo; ep_mod = np.sqrt(np.diag(np.asarray(mod_c.cov_params(), float)))
LC = []
for ck in [50, 100, 150, 200, 250, 300, 400, 500, 600, 800]:
    e = np.sqrt(np.diag(np.asarray(S.cov_conley(mod_c, lat, lon, ck), float)))
    for j, nm in enumerate(nomes_c): LC.append(dict(param=nm, coef=float(mod_c.params[j]), corte_km=ck, ep=float(e[j]), razao=float(e[j] / ep_mod[j])))
CURVA = pd.DataFrame(LC); CURVA.to_csv(TAB / "final_ep_corte.csv", index=False)
print(CURVA.pivot(index="param", columns="corte_km", values="razao").round(2).to_string())
FIN.update({"corte_2026": CORTE_2026, "corte_2022": CORTE_2022, "corte_final": CORTE, "motivo_corte": MOTIVO_CORTE})
(TAB / "final_decisao.json").write_text(json.dumps(FIN, indent=2, ensure_ascii=False, default=str))

  ini_km     fim_km   pares  correlacao
  0.0000    10.0000     617      0.4781
 10.0000    25.0000   10314      0.4075
 25.0000    50.0000   35450      0.2891
 50.0000    75.0000   53238      0.2199
 75.0000   100.0000   68805      0.1759
100.0000   150.0000  176025      0.1165
150.0000   200.0000  217585      0.0836
200.0000   300.0000  531651      0.0293
300.0000   400.0000  609990     -0.0340
400.0000   600.0000 1297109     -0.0355
600.0000   800.0000 1383337      0.0028
800.0000 1,000.0000 1378747     -0.0127
corte 2026 = 200 km | corte 2022 = 200 km | corte final = 200 km | CORTE_2022=200 km dentro de [100, 400]; corte de 2026 mantido


corte_km                 50     100    150    200    250    300    400    500    600    800
param                                                                                      
bloco_esquerda        1.1300 1.2400 1.3200 1.3800 1.4800 1.5700 1.7700 1.9200 2.0100 2.1700
bloco_outro           0.9800 0.9700 0.9500 0.9500 0.9600 0.9600 0.9200 0.8900 0.8900 0.8800
cobertura_pbf         1.0500 1.1200 1.2300 1.3300 1.4200 1.5100 1.6800 1.8300 1.9500 2.0900
comparecimento        1.0300 1.0400 1.0900 1.1600 1.2500 1.3500 1.5300 1.6600 1.7700 1.8800
const                 1.3900 1.7600 2.0100 2.1700 2.3000 2.4100 2.5400 2.6000 2.6200 2.6100
disp_const            1.4600 1.8800 2.2200 2.4500 2.5900 2.6500 2.6500 2.6300 2.7100 2.8000
disp_log_n            1.3300 1.5000 1.6200 1.7500 1.8700 1.9800 2.1000 2.1700 2.2400 2.3600
disp_reg_Centro-Oeste 1.2500 1.4700 1.6500 1.8100 1.9300 2.0200 2.1400 2.1900 2.2300 2.1900
disp_reg_Nordeste     1.4800 1.8100 2.1000 2.2900 2.4000 2.4400 2.4700 2.4800 2.

7098

**Leitura.** O correlograma de 2022 cai abaixo de 0,05 no mesmo ponto que o de 2026: o corte de 2022 é 200 km, dentro de [100, 400]. O corte de 2026 (200 km) é mantido. A correlação de 2022 é maior a curta distância (0,48 contra 0,38 até 10 km), mas o alcance é o mesmo. A razão Conley/modelo cresce com o corte (intercepto: 1,39 a 50 km, 2,17 a 200 km, 2,61 a 800 km), então a escolha do corte importa menos que usar algum corte.

*Números da execução de 2026-11-18; se o notebook for reexecutado, confira as saídas acima.*

In [12]:
mod = AF.modelo
nomes_p = list(AF.nomes); assert len(nomes_p) == len(mod.params), (len(nomes_p), len(mod.params))
def ep(cov): return np.sqrt(np.diag(np.asarray(cov, float)))
EP = {"ep_modelo": ep(mod.cov_params())}
for tag, f in [("ep_conley_05", 0.5), ("ep_conley_1", 1.0), ("ep_conley_2", 2.0)]:
    EP[tag] = ep(S.cov_conley(mod, lat, lon, CORTE * f))
EP["ep_cluster_uf"] = ep(S.cov_cluster(mod, uf, correcao=True))
TB = pd.DataFrame({"param": nomes_p, "tipo": ["beta"] * NB_ + ["gamma"] * (len(nomes_p) - NB_), "coef": np.asarray(mod.params, float), **EP})
TB["razao_conley_modelo"] = TB.ep_conley_1 / TB.ep_modelo
TB["signif_modelo"] = (TB.coef / TB.ep_modelo).abs() > 1.96
TB["signif_conley"] = (TB.coef / TB.ep_conley_1).abs() > 1.96
TB["muda"] = TB.signif_modelo != TB.signif_conley
TB.to_csv(TAB / "final_ep.csv", index=False)
print(f"corte {CORTE:g} km (×0,5 = {CORTE*.5:g}; ×2 = {CORTE*2:g})")
print(TB.to_string(index=False))
rb = TB[TB.tipo == "beta"].razao_conley_modelo
print(f"\nrazão Conley/modelo (β): mediana {rb.median():.2f} | mín {rb.min():.2f} | máx {rb.max():.2f}")
print("coeficientes que mudam de significância:", list(TB[TB.muda].param) or "nenhum")

corte 200 km (×0,5 = 100; ×2 = 400)
                param  tipo    coef  ep_modelo  ep_conley_05  ep_conley_1  ep_conley_2  ep_cluster_uf  razao_conley_modelo  signif_modelo  signif_conley  muda
                const  beta  0.1945     0.0032        0.0057       0.0070       0.0082         0.0096               2.1721           True           True False
                  lag  beta  0.7749     0.0041        0.0046       0.0051       0.0058         0.0069               1.2537           True           True False
          pct_dep_esq  beta  0.0097     0.0017        0.0018       0.0020       0.0025         0.0025               1.2103           True           True False
       comparecimento  beta -0.0175     0.0019        0.0020       0.0022       0.0029         0.0030               1.1607           True           True False
             log_dens  beta  0.0149     0.0028        0.0030       0.0035       0.0042         0.0043               1.2151           True           True False
         p

**Leitura.** `razao_conley_modelo` > 1 indica que o EP do modelo subestima a incerteza. O corte foi fixado pela regra, antes de ver os EPs; as colunas ×0,5 e ×2 e o conglomerado por UF mostram a sensibilidade.

No modelo final, a razão Conley/modelo dos β tem mediana 1,69 (de 0,95 a 3,49). Quatro coeficientes perdem significância com Conley: `w_pct_vab_agro`, `bloco_esquerda`, `reg_Centro-Oeste` e `reg_Nordeste+Norte`. Os termos da dispersão também ficam mais incertos (razão de 1,75 a 2,56). As conclusões sobre esses termos devem usar o EP de Conley.

*Números da execução de 2026-11-18; se o notebook for reexecutado, confira as saídas acima.*

## Parte 16 · Estimativas regionais e por UF do modelo final (LOSO)

**Procedimento** (o mesmo do notebook 03, Parte 11). Para cada UF, o modelo final é reajustado sem ela e a prevê.
Intervalos: B = 1.000 sorteios por UF retida (semente 7, quantis 2,5/50/97,5 %; guardamos também 5/95 %) de
(i) β ~ N(β̂, Σ̂), com Σ̂ o bloco de β da covariância do ajuste sem a UF; (ii) choque de UF δ ~ N(0, σ²) no logit;
(iii) votos ~ BetaBin(n, π, θᵢ) com **θᵢ por município** (`prever_theta`, dispersão por região e log n).

**Duas covariâncias** são comparadas: a **Conley** (corte da Parte 15, calculada no ajuste de treino de cada UF)
e a **do modelo**. **Dois σ**: 0 e σ_UF = RMS dos resíduos LOSO das UFs no logit (receita do notebook 03).

**Aviso.** σ_UF é estimado nas mesmas 27 UFs em que a cobertura é conferida, então a cobertura com σ_UF é
**otimista**. A cobertura com σ = 0 não usa essa informação, mas ignora o erro de extrapolar para uma UF inédita.

In [13]:
from scipy.special import logit
def parametros_f(nome_uf):
    tr = np.flatnonzero(uf != nome_uf); te = np.flatnonzero(uf == nome_uf)
    a = CF.ajustar(df.iloc[tr], mapa)
    dte = df.iloc[te].copy(); dte[mapa["n"]] = NHAT[te]   # UF retida não apurada: peso n estimado
    X = E.montar_X(dte, mapa, a.pad, CF.termos)
    nb_ = a.extra["n_beta"]; assert X.shape[1] == nb_
    th = E.prever_theta(a, dte, mapa)
    cm = np.asarray(a.modelo.cov_params(), float)[:nb_, :nb_]
    cc = np.asarray(S.cov_conley(a.modelo, lat[tr], lon[tr], CORTE), float)[:nb_, :nb_]
    return dict(beta=np.asarray(a.modelo.params, float)[:nb_], cov_modelo=cm, cov_conley=cc, X=X.to_numpy(float), theta=th, te=te)
NHAT = E.n_loso(df)
t0 = time.time()
with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
    PARF = dict(zip(UFS_, ex.map(parametros_f, UFS_)))
print("ajustes LOSO do modelo final:", NOME_F, round(time.time() - t0), "s")
pi_loso_f = PIL[NOME_F]
prev_uf_f = M.agregar(pi_loso_f, NHAT, uf).drop("Brasil")
res_logit_f = logit(prev_uf_f[UFS_].values) - logit(obs_uf[UFS_].values)
SIG_F = float(np.sqrt(np.mean(res_logit_f ** 2)))
print(f"σ_UF (RMS dos resíduos LOSO no logit) = {SIG_F:.4f} | viés médio = {res_logit_f.mean():+.4f}")
print("razão média diag(cov Conley)/diag(cov modelo) do bloco β, por UF retida: mediana",
      round(float(np.median([np.mean(np.diag(v['cov_conley']) / np.diag(v['cov_modelo'])) for v in PARF.values()])), 2))

ajustes LOSO do modelo final: bb_slx 37 s
σ_UF (RMS dos resíduos LOSO no logit) = 0.0564 | viés médio = -0.0235
razão média diag(cov Conley)/diag(cov modelo) do bloco β, por UF retida: mediana 2.96


In [14]:
n_uf = pd.Series(NHAT).groupby(uf).sum()[UFS_].to_numpy(); uf_reg = pd.Series(reg).groupby(uf).first()[UFS_].to_numpy()
def simular_f(chave, sigma, B=1000, seed=7):
    rng = np.random.default_rng(seed)
    return np.column_stack([M.simular_votos_grupo(v["beta"], v[chave], v["X"], NHAT[v["te"]], np.zeros(len(v["te"])), link=E.LIGACOES[CF.ligacao],
                            theta=v["theta"], sigma=sigma, B=B, rng=rng)[:, 0] for v in PARF.values()])
def resumo_f(chave, sigma):
    V = simular_f(chave, sigma); linhas = {}
    for i, u in enumerate(UFS_): linhas[("uf", u)] = (V[:, i], n_uf[i], obs_uf[u])
    for r in REGS:
        m = uf_reg == r; linhas[("regiao", r)] = (V[:, m].sum(1), n_uf[m].sum(), obs_reg[r])
    linhas[("regiao", "Brasil")] = (V.sum(1), n_uf.sum(), y.sum() / n.sum())
    out = []
    for (nivel, nome), (v, nn, real) in linhas.items():
        q = np.quantile(v / nn, [0.025, 0.05, 0.5, 0.95, 0.975])
        pt = (prev_uf_f[UFS_].values * n_uf)[uf_reg == nome].sum() / nn if nivel == "regiao" and nome != "Brasil" else \
             (prev_uf_f[nome] if nivel == "uf" else (prev_uf_f[UFS_].values * n_uf).sum() / nn)
        out.append(dict(nivel=nivel, nome=nome, lula_pct=pt * 100, q025=q[0] * 100, q05=q[1] * 100, q50=q[2] * 100, q95=q[3] * 100, q975=q[4] * 100, real_pct=real * 100))
    d = pd.DataFrame(out); d["erro_pp"] = d.lula_pct - d.real_pct
    d["dentro_ic95"] = (d.real_pct >= d.q025) & (d.real_pct <= d.q975); d["dentro_ic90"] = (d.real_pct >= d.q05) & (d.real_pct <= d.q95)
    d["largura_pp"] = d.q975 - d.q025; d["largura90_pp"] = d.q95 - d.q05
    return d
VARS = {("conley", "sigma_uf"): ("cov_conley", SIG_F), ("conley", "zero"): ("cov_conley", 0.0),
        ("modelo", "sigma_uf"): ("cov_modelo", SIG_F), ("modelo", "zero"): ("cov_modelo", 0.0)}
EST_F = pd.concat([resumo_f(ch, s).assign(covariancia=cv, sigma=sg) for (cv, sg), (ch, s) in VARS.items()], ignore_index=True)
col0 = ["covariancia", "sigma"]
EST_F[EST_F.nivel == "regiao"].drop(columns="nivel").rename(columns={"nome": "regiao"}).to_csv(TAB / "final_estimativas_regionais.csv", index=False)
EST_F[EST_F.nivel == "uf"].drop(columns="nivel").rename(columns={"nome": "uf"}).to_csv(TAB / "final_estimativas_uf.csv", index=False)
COB = []
for (cv, sg), d in EST_F.groupby(col0):
    u, r = d[d.nivel == "uf"], d[(d.nivel == "regiao") & (d.nome != "Brasil")]
    COB.append(dict(covariancia=cv, sigma=sg, valor_sigma=SIG_F if sg == "sigma_uf" else 0.0, cob95_uf=u.dentro_ic95.mean(), n_uf_dentro95=int(u.dentro_ic95.sum()),
                    cob90_uf=u.dentro_ic90.mean(), cob95_regiao=r.dentro_ic95.mean(), n_reg_dentro95=int(r.dentro_ic95.sum()), cob90_regiao=r.dentro_ic90.mean(),
                    largura95_uf_pp=u.largura_pp.mean(), largura95_reg_pp=r.largura_pp.mean()))
COB = pd.DataFrame(COB); COB.to_csv(TAB / "final_cobertura.csv", index=False)
print(COB.to_string(index=False)); print("(nominal 95 % e 90 %; 27 UFs, 5 regiões; cobertura com σ_UF é otimista)")
PF = EST_F[(EST_F.covariancia == "conley") & (EST_F.sigma == "sigma_uf") & (EST_F.nivel == "regiao")]
print(PF[["nome", "lula_pct", "q05", "q95", "q025", "q975", "real_pct", "erro_pp", "dentro_ic95"]].to_string(index=False))

covariancia    sigma  valor_sigma  cob95_uf  n_uf_dentro95  cob90_uf  cob95_regiao  n_reg_dentro95  cob90_regiao  largura95_uf_pp  largura95_reg_pp
     conley sigma_uf       0.0564    1.0000             27    1.0000        1.0000               5        1.0000           5.2981            2.8251
     conley     zero       0.0000    0.5185             14    0.4074        0.4000               2        0.4000           2.0395            0.8677
     modelo sigma_uf       0.0564    1.0000             27    0.9259        1.0000               5        1.0000           5.2571            2.7949
     modelo     zero       0.0000    0.3333              9    0.3333        0.4000               2        0.4000           1.7413            0.6941
(nominal 95 % e 90 %; 27 UFs, 5 regiões; cobertura com σ_UF é otimista)
        nome  lula_pct     q05     q95    q025    q975  real_pct  erro_pp  dentro_ic95
Centro-Oeste   40.0903 38.8194 41.3892 38.5838 41.6229   39.9912   0.0991         True
    Nordeste  

In [15]:
pf = PF.set_index("nome")
BASE_PT = M.agregar(PIL["bb_selecao"], NHAT, reg).mul(100).reindex(pf.index)
CMP = pd.DataFrame({"real_pct": pf.real_pct, "base_pct": BASE_PT, "base_erro_pp": BASE_PT - pf.real_pct,
    "final_pct": pf.lula_pct, "final_q025": pf.q025, "final_q975": pf.q975, "final_q05": pf.q05, "final_q95": pf.q95, "final_erro_pp": pf.erro_pp, "final_largura_pp": pf.largura_pp}).rename_axis("regiao")
CMP.to_csv(TAB / "final_comparacao_regional.csv"); print(CMP.round(3).to_string())
print("erro absoluto médio nas 5 regiões (pp): bb_selecao", round(CMP.drop("Brasil").base_erro_pp.abs().mean(), 3), "| final", round(CMP.drop("Brasil").final_erro_pp.abs().mean(), 3))
d = PF.sort_values("lula_pct")
fig = go.Figure()
fig.add_trace(go.Scatter(x=BASE_PT.loc[d.nome].values, y=d.nome, mode="markers", name="bb_selecao (ponto)", marker=dict(color="#888", size=8)))
fig.add_trace(go.Scatter(x=d.lula_pct, y=d.nome, mode="markers", name=f"{NOME_F}, Conley (IC95)", error_x=dict(type="data", symmetric=False, array=d.q975 - d.lula_pct, arrayminus=d.lula_pct - d.q025), marker=dict(color="#1b6ca8", size=9)))
fig.add_trace(go.Scatter(x=d.real_pct, y=d.nome, mode="markers", name="real", marker=dict(color="#c2571a", symbol="diamond", size=10)))
fig.update_layout(title="Share de Lula por região (LOSO, σ_UF): bb_selecao e modelo final", xaxis_title="% dos votos válidos", template="plotly_white", height=460)
G.salvar(fig, FIG / "final_regional.html"); fig.show()
print("tempo total do notebook:", round(time.time() - T0), "s")

              real_pct  base_pct  base_erro_pp  final_pct  final_q025  final_q975  final_q05  final_q95  final_erro_pp  final_largura_pp
regiao                                                                                                                                  
Centro-Oeste   39.9910   40.3800        0.3890    40.0900     38.5840     41.6230    38.8190    41.3890         0.0990            3.0390
Nordeste       69.3690   69.3550       -0.0130    69.4360     68.4850     70.2900    68.6640    70.1790         0.0670            1.8040
Norte          50.0050   49.4420       -0.5630    49.1850     47.6770     50.7200    47.9200    50.4960        -0.8210            3.0440
Sudeste        47.2420   47.6460        0.4040    47.8900     46.3280     49.5590    46.5070    49.2830         0.6480            3.2300
Sul            37.4240   36.8590       -0.5640    36.9290     35.4480     38.4560    35.6700    38.1320        -0.4950            3.0090
Brasil         51.7800   51.6740       -0

tempo total do notebook: 243 s


**Leitura.** Com o LOSO do modelo final (UF retida com peso n̂, sem o 2º turno dela), o erro médio nas 5 regiões é 0,43 p.p. (0,39 em `bb_selecao`), e o Brasil erra 0,02 p.p. (51,76% contra 51,78% real; `bb_selecao` 51,67%). Os maiores erros são Norte (−0,82), Sudeste (+0,65) e Sul (−0,49). Os intervalos de 95% (Conley, σ_UF = 0,0564) têm largura de 1,5 p.p. no Brasil, 1,8 no Nordeste e 3,0 a 3,2 nas demais regiões. Com 5 regiões, a comparação com `bb_selecao` é descrição, não evidência de superioridade ([doc 09](../docs/pressupostos/09-modelo-final.md), seção 5). A cobertura com σ_UF (27/27 UFs e 5/5 regiões) é otimista, porque σ_UF é ajustado nas mesmas UFs; com σ = 0 a cobertura cai para 52% das UFs (14/27) e 40% das regiões (2/5): o erro de extrapolar para uma UF nova pesa mais que a incerteza dos coeficientes.

*Números da execução de 2026-11-18; se o notebook for reexecutado, confira as saídas acima.*

## Parte 17 · Extensões rejeitadas: krigagem e CAR/BYM

O modelo final da Parte 14 (`modelo_final`) ainda deixa correlação espacial nos resíduos. Esta Parte (a) mede onde a correlação mora e testa duas correções aplicadas aos resíduos, (b) a krigagem e (c) o efeito aleatório CAR/BYM, ambas com a mesma regra de aceitação fixada antes do resultado.

**Regra (escrita em 2026-11-09, antes de ver o resultado).**

1. **18a, diagnóstico, sem decisão.** Com os resíduos quantílicos de `modelo_final` ajustado em todos os dados: I de Moran com o W completo; com W só com vínculos dentro da mesma UF; com W só com vínculos entre UFs diferentes (padronizados por linha; linha sem vínculo sai); e com o W completo depois de subtrair a média de cada UF dos resíduos.
2. **18b, candidato `modelo_final_krig`.** Em cada dobra LOSO (treino = 26 UFs, teste = 1 UF): ajusta `modelo_final` no treino; resíduo de treino r = logit((y+0,5)/(n+1)) − η̂. Para cada município i do teste, k̄ᵢ = Σⱼ K(dᵢⱼ) rⱼ / Σⱼ K(dᵢⱼ) sobre o treino, com K de Bartlett = 1 − d/`CORTE` km para d < `CORTE` km (corte = `CORTE`, sem ajuste); k̄ᵢ = 0 se não houver vizinho do treino a menos de `CORTE` km. O encolhimento ρ̂ vem só do treino: inclinação por MQO sem intercepto de rᵢ em k̄ᵢ, em que k̄ᵢ de um município de treino usa só municípios de treino de OUTRAS UFs; ρ̂ cortado em [0, 1]. Previsão: π̃ᵢ = expit(logit(π̂ᵢ) + ρ̂·k̄ᵢ). O 5-fold repete o procedimento com as dobras de 5 (UF "outra" continua sendo outra UF) e é só descritivo.
3. **Aceitação (fixa).** Aceita `modelo_final_krig` se, ao mesmo tempo: (1) o I de Moran (W completo) dos resíduos logit fora da amostra do LOSO, logit((y+0,5)/(n+1)) − logit(π̃), cai pelo menos 0,05 em relação ao mesmo número para `modelo_final` (`PIL["bb_slx"]`); e (2) o MAE por UF do LOSO de `modelo_final_krig` é no máximo o `limite` de `final_decisao.json` (1,2515 p.p.). Se não passar, o item fecha como limitação documentada.
4. Mesmo se aceito, esta Parte não muda `modelo_final` nem a Parte 16: só reporta.

In [16]:
mor_dec = S.moran_decomposto(RF, W, uf)
DEC_T = pd.DataFrame({"w": ["completo", "dentro_uf", "entre_uf", "completo_sem_media_uf"],
                      "I_moran": [mor_dec[k] for k in ["completo", "dentro_uf", "entre_uf", "completo_sem_media_uf"]],
                      "n_linhas": [mor_dec[f"n_{k}"] for k in ["completo", "dentro_uf", "entre_uf", "completo_sem_media_uf"]]})
DEC_T.to_csv(TAB / "final_moran_decomposto.csv", index=False)
print(DEC_T.to_string(index=False))

                    w  I_moran  n_linhas
             completo   0.3361      5570
            dentro_uf   0.3439      5568
             entre_uf   0.1570      1132
completo_sem_media_uf   0.2683      5570


In [17]:
from scipy.special import logit, expit
CK = CF
lz = logit((y + 0.5) / (n + 1))
def _krig_fold(args):
    tr, te = args
    a = CK.ajustar(df.iloc[tr], mapa)
    pi_tr = np.asarray(CK.prever(a, df.iloc[tr], mapa), float)
    rho, k = S.krigagem_residuos(lz[tr] - logit(pi_tr), lat, lon, tr, te, uf, CORTE)
    pi_te = np.asarray(CK.prever(a, df.iloc[te], mapa), float)
    return rho, expit(logit(pi_te) + rho * k)
def rodar_krig(esq, folds):
    arq = CV / f"{esq}_{NOME_F}_krig.parquet"
    if arq.exists() and not RECALCULAR:
        d = pd.read_parquet(arq); return d["pi"].to_numpy(), d["rho"].to_numpy()
    with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
        res = list(ex.map(_krig_fold, folds))
    pi, rho = np.full(len(df), np.nan), np.full(len(df), np.nan)
    for (tr, te), (r_, p_) in zip(folds, res): pi[te], rho[te] = p_, r_
    pd.DataFrame({"pi": pi, "rho": rho}).to_parquet(arq); return pi, rho
PIK5, RHO5 = rodar_krig("kfold", folds5)
PIKL, RHOL = rodar_krig("loso", foldsL)
print("NaN:", int(np.isnan(PIK5).sum()), int(np.isnan(PIKL).sum()), "|", round(time.time() - T0), "s")
def rho_por_dobra(rho, folds): return np.array([rho[te][0] for _, te in folds])
def moran_oos(pi): return S.moran_residuos(lz - logit(pi), W, permutacoes=999, semente=0)["I"]
LK = []
for esq, folds, base, krig, rho in [("5-fold", folds5, PI5[NOME_F], PIK5, RHO5), ("LOSO", foldsL, PIL[NOME_F], PIKL, RHOL)]:
    rf = rho_por_dobra(rho, folds)
    LK.append({"esquema": esq, "modelo": NOME_F, **erros(base, esq, folds), "moran_oos": moran_oos(base), "rho_media": np.nan, "rho_min": np.nan, "rho_max": np.nan})
    LK.append({"esquema": esq, "modelo": NOME_F + "_krig", **erros(krig, esq, folds), "moran_oos": moran_oos(krig), "rho_media": rf.mean(), "rho_min": rf.min(), "rho_max": rf.max()})
KRIG_T = pd.DataFrame(LK); KRIG_T.to_csv(TAB / "final_krigagem.csv", index=False)
print(KRIG_T[["esquema", "modelo", "mae_uf_pp", "se_uf_pp", "mae_pond_pp", "mae_reg_pp", "moran_oos", "rho_media", "rho_min", "rho_max"]].to_string(index=False))

NaN: 0 0 | 243 s


esquema      modelo  mae_uf_pp  se_uf_pp  mae_pond_pp  mae_reg_pp  moran_oos  rho_media  rho_min  rho_max
 5-fold      bb_slx     0.8459    0.1501       1.1483      0.3406     0.3207        NaN      NaN      NaN
 5-fold bb_slx_krig     0.8435    0.1494       1.1473      0.3402     0.3198     0.0037   0.0000   0.0187
   LOSO      bb_slx     1.0777    0.1497       1.4062      0.4260     0.4224        NaN      NaN      NaN
   LOSO bb_slx_krig     1.0771    0.1499       1.4060      0.4283     0.4222     0.0050   0.0000   0.0408


In [18]:
kl = KRIG_T[KRIG_T.esquema == "LOSO"].set_index("modelo")
I_antes, I_depois = float(kl.loc[NOME_F, "moran_oos"]), float(kl.loc[NOME_F + "_krig", "moran_oos"])
MAE_K, LIM = float(kl.loc[NOME_F + "_krig", "mae_uf_pp"]), float(FIN["limite"])
c1, c2 = bool(I_antes - I_depois >= 0.05), bool(MAE_K <= LIM)
KRIG_OK = c1 and c2
k5 = KRIG_T[KRIG_T.esquema == "5-fold"].set_index("modelo")
print(f"krigagem {'ACEITA' if KRIG_OK else 'REJEITADA'}: queda do I de Moran LOSO {I_antes:.4f} -> {I_depois:.4f} (queda {I_antes - I_depois:+.4f}; exige >= 0,05: {c1}) | "
      f"MAE por UF LOSO {MAE_K:.4f} <= limite {LIM:.4f}: {c2}")
dec = json.loads((TAB / "final_decisao.json").read_text())
dec["aceitas"]["krig"] = KRIG_OK
dec["krigagem"] = {"regra": {"queda_minima_moran": 0.05, "mae_uf_max": LIM, "corte_km": CORTE},
                   "moran_oos_loso": {NOME_F: I_antes, NOME_F + "_krig": I_depois, "queda": I_antes - I_depois},
                   "moran_oos_5fold": {m: float(k5.loc[m, "moran_oos"]) for m in k5.index},
                   "mae_uf_loso": {m: float(kl.loc[m, "mae_uf_pp"]) for m in kl.index},
                   "mae_uf_5fold": {m: float(k5.loc[m, "mae_uf_pp"]) for m in k5.index},
                   "rho_loso": {c: float(kl.loc[NOME_F + "_krig", f"rho_{c}"]) for c in ["media", "min", "max"]},
                   "rho_5fold": {c: float(k5.loc[NOME_F + "_krig", f"rho_{c}"]) for c in ["media", "min", "max"]},
                   "criterio_moran_ok": c1, "criterio_mae_ok": c2, "aceita": KRIG_OK}
(TAB / "final_decisao.json").write_text(json.dumps(dec, indent=2, ensure_ascii=False, default=str))
print("modelo_final no JSON (inalterado):", dec["modelo_final"])

krigagem REJEITADA: queda do I de Moran LOSO 0.4224 -> 0.4222 (queda +0.0003; exige >= 0,05: False) | MAE por UF LOSO 1.0771 <= limite 1.2515: True
modelo_final no JSON (inalterado): bb_slx


**Leitura.** A correlação mora dentro das UFs: o I de Moran dos resíduos é 0,336 com o W completo, 0,344 só com vínculos dentro da UF e 0,157 só com vínculos entre UFs (1.132 municípios de fronteira). Tirar a média de cada UF baixa o I só para 0,268, então o choque comum de UF explica pouco; o resto é correlação local entre vizinhos. A krigagem dos resíduos não ajuda fora da amostra: o encolhimento ρ̂ fica perto de zero (média 0,005 no LOSO, de 0 a 0,041), porque o resíduo de um município quase não prevê o de outra UF. O I de Moran dos resíduos LOSO vai de 0,4224 (`bb_slx`) para 0,4222 (`bb_slx_krig`), queda de 0,0003 contra os 0,05 exigidos. O MAE por UF no LOSO é 1,077 p.p. (igual ao de `bb_slx`, 1,078), dentro do limite de 1,2515; no 5-fold é 0,844 (0,846 em `bb_slx`). Pela regra fixada antes, `bb_slx_krig` é rejeitado (o critério 1 falha) e o item fecha como limitação documentada. `modelo_final` continua `bb_slx`.

*Números da execução de 2026-11-18; se o notebook for reexecutado, confira as saídas acima.*

### Efeito espacial CAR/BYM nos resíduos (LOSO)

A segunda alternativa espacial é um efeito aleatório BYM em dois estágios sobre os resíduos de `modelo_final`, sem dependência nova.

**Regra (escrita em 2026-11-09, antes de ver o resultado; igual à da krigagem).**

1. Em cada dobra (LOSO e 5-fold), ajusta `modelo_final` no treino; resíduo logit r = logit((y+0,5)/(n+1)) − η̂. Modelo: r = u + ε, u CAR de Leroux com precisão (1/τ)[(1−λ)I + λ(D−A)] na adjacência queen binária, ε ~ N(0, σ²). (τ, λ, σ²) por máxima verossimilhança gaussiana só no subgrafo de treino; λ limitado a 0,99.
2. Previsão: média a posteriori de u no grafo completo dado só r de treino; π̃ = expit(η̂ + û).
3. **Aceitação (fixa).** Aceita `modelo_final_car` se, ao mesmo tempo: (1) o I de Moran dos resíduos logit LOSO cai pelo menos 0,05 em relação a `modelo_final`; e (2) o MAE por UF do LOSO é no máximo o `limite` de `final_decisao.json`. `modelo_final` só muda se os dois passarem.

In [19]:
def _car_fold(args):
    tr, te = args
    a = CK.ajustar(df.iloc[tr], mapa)
    pi_tr = np.asarray(CK.prever(a, df.iloc[tr], mapa), float)
    par, u = S.car_residuos(lz[tr] - logit(pi_tr), W, tr, te)
    pi_te = np.asarray(CK.prever(a, df.iloc[te], mapa), float)
    return par, expit(logit(pi_te) + u)
def rodar_car(esq, folds):
    arq = CV / f"{esq}_{NOME_F}_car.parquet"
    if arq.exists() and not RECALCULAR:
        d = pd.read_parquet(arq); return d["pi"].to_numpy(), d[["tau", "lam", "sigma2"]].to_numpy()
    with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
        res = list(ex.map(_car_fold, folds))
    pi, P = np.full(len(df), np.nan), np.full((len(df), 3), np.nan)
    for (tr, te), (par, p_) in zip(folds, res): pi[te], P[te] = p_, [par["tau"], par["lam"], par["sigma2"]]
    pd.DataFrame({"pi": pi, "tau": P[:, 0], "lam": P[:, 1], "sigma2": P[:, 2]}).to_parquet(arq); return pi, P
PIC5, PAR5 = rodar_car("kfold", folds5)
PICL, PARL = rodar_car("loso", foldsL)
print("NaN:", int(np.isnan(PIC5).sum()), int(np.isnan(PICL).sum()), "|", round(time.time() - T0), "s")
def par_por_dobra(P, folds): return np.array([P[te][0] for _, te in folds])
LC = []
for esq, folds, base, car, P in [("5-fold", folds5, PI5[NOME_F], PIC5, PAR5), ("LOSO", foldsL, PIL[NOME_F], PICL, PARL)]:
    pf = par_por_dobra(P, folds)
    LC.append({"esquema": esq, "modelo": NOME_F, **erros(base, esq, folds), "moran_oos": moran_oos(base)})
    LC.append({"esquema": esq, "modelo": NOME_F + "_car", **erros(car, esq, folds), "moran_oos": moran_oos(car),
               **{f"{k}_{s}": f(pf[:, j]) for j, k in enumerate(["tau", "lam", "sigma2"]) for s, f in [("media", np.mean), ("min", np.min), ("max", np.max)]}})
CAR_T = pd.DataFrame(LC); CAR_T.to_csv(TAB / "final_car.csv", index=False)
print(CAR_T[["esquema", "modelo", "mae_uf_pp", "se_uf_pp", "mae_pond_pp", "mae_reg_pp", "moran_oos", "tau_media", "lam_media", "sigma2_media"]].to_string(index=False))
print(CAR_T[["esquema", "modelo", "tau_min", "tau_max", "lam_min", "lam_max", "sigma2_min", "sigma2_max"]].to_string(index=False))

NaN: 0 0 | 244 s


esquema     modelo  mae_uf_pp  se_uf_pp  mae_pond_pp  mae_reg_pp  moran_oos  tau_media  lam_media  sigma2_media
 5-fold     bb_slx     0.8459    0.1501       1.1483      0.3406     0.3207        NaN        NaN           NaN
 5-fold bb_slx_car     0.4736    0.1193       0.9207      0.2331    -0.0429     0.0096     0.8108        0.0031
   LOSO     bb_slx     1.0777    0.1497       1.4062      0.4260     0.4224        NaN        NaN           NaN
   LOSO bb_slx_car     1.0037    0.1403       1.4051      0.5634     0.3949     0.0153     0.7981        0.0023
esquema     modelo  tau_min  tau_max  lam_min  lam_max  sigma2_min  sigma2_max
 5-fold     bb_slx      NaN      NaN      NaN      NaN         NaN         NaN
 5-fold bb_slx_car   0.0079   0.0128   0.4950   0.9140      0.0014      0.0040
   LOSO     bb_slx      NaN      NaN      NaN      NaN         NaN         NaN
   LOSO bb_slx_car   0.0136   0.0176   0.4950   0.9354      0.0008      0.0031


In [20]:
cl = CAR_T[CAR_T.esquema == "LOSO"].set_index("modelo")
IC_antes, IC_depois = float(cl.loc[NOME_F, "moran_oos"]), float(cl.loc[NOME_F + "_car", "moran_oos"])
MAE_C = float(cl.loc[NOME_F + "_car", "mae_uf_pp"])
d1, d2 = bool(IC_antes - IC_depois >= 0.05), bool(MAE_C <= LIM)
CAR_OK = d1 and d2
c5 = CAR_T[CAR_T.esquema == "5-fold"].set_index("modelo")
print(f"CAR {'ACEITO' if CAR_OK else 'REJEITADO'}: queda do I de Moran LOSO {IC_antes:.4f} -> {IC_depois:.4f} (queda {IC_antes - IC_depois:+.4f}; exige >= 0,05: {d1}) | "
      f"MAE por UF LOSO {MAE_C:.4f} <= limite {LIM:.4f}: {d2}")
dec = json.loads((TAB / "final_decisao.json").read_text())
dec["aceitas"]["car"] = CAR_OK
par_cols = [f"{k}_{s}" for k in ["tau", "lam", "sigma2"] for s in ["media", "min", "max"]]
dec["car"] = {"regra": {"queda_minima_moran": 0.05, "mae_uf_max": LIM},
              "moran_oos_loso": {NOME_F: IC_antes, NOME_F + "_car": IC_depois, "queda": IC_antes - IC_depois},
              "moran_oos_5fold": {m: float(c5.loc[m, "moran_oos"]) for m in c5.index},
              "mae_uf_loso": {m: float(cl.loc[m, "mae_uf_pp"]) for m in cl.index},
              "mae_uf_5fold": {m: float(c5.loc[m, "mae_uf_pp"]) for m in c5.index},
              "parametros_loso": {c: float(cl.loc[NOME_F + "_car", c]) for c in par_cols},
              "parametros_5fold": {c: float(c5.loc[NOME_F + "_car", c]) for c in par_cols},
              "criterio_moran_ok": d1, "criterio_mae_ok": d2, "aceita": CAR_OK}
(TAB / "final_decisao.json").write_text(json.dumps(dec, indent=2, ensure_ascii=False, default=str))
print("modelo_final no JSON:", dec["modelo_final"])

CAR REJEITADO: queda do I de Moran LOSO 0.4224 -> 0.3949 (queda +0.0275; exige >= 0,05: False) | MAE por UF LOSO 1.0037 <= limite 1.2515: True
modelo_final no JSON: bb_slx


**Leitura.** O CAR em dois estágios melhora o erro por UF, mas não tira a correlação espacial dos resíduos. No LOSO, o MAE por UF cai de 1,078 para 1,004 p.p. (limite 1,2515: critério 2 passa), mas o MAE regional sobe de 0,426 para 0,563 p.p. O I de Moran dos resíduos LOSO vai de 0,4224 para 0,3949, queda de 0,0275 contra os 0,05 exigidos (critério 1 falha). Parâmetros do treino (média das 27 dobras LOSO): τ = 0,0153, λ = 0,798 (0,495 a 0,935) e σ² = 0,0023; o efeito estruturado domina o ruído iid. No 5-fold, onde cada município de teste tem vizinhos de treino, o MAE por UF cai de 0,846 para 0,474 p.p. e o I vai a −0,043: o CAR interpola bem entre vizinhos, mas uma UF inteira deixada de fora só é informada pelos municípios da divisa. Pela regra fixada antes, `bb_slx_car` é rejeitado e `modelo_final` continua `bb_slx`.

*Números da execução de 2026-11-18; se o notebook for reexecutado, confira as saídas acima.*

### Sensibilidade ponderada: krigagem e CAR em resíduos de Pearson

**Motivo.** Os resíduos logit usados acima não têm variância comum: a da proporção y/n é cerca de 1/n, então municípios pequenos pesam tanto quanto os grandes na estimativa de ρ̂, no CAR e no I de Moran. Esta subseção repete as duas análises com resíduos padronizados. É **informativa**: as decisões da krigagem e do CAR não mudam.

**Especificação (fixada em 2026-11-22, antes de ver o resultado).**

1. Resíduo de Pearson beta-binomial z = (y/n − π̂) / sd, sd² = π̂(1 − π̂)[1 + (n − 1)θ]/n, com θ por município da dispersão do ajuste da dobra (`prever_theta`). Mesmo π̂ e mesmo n dos resíduos atuais: π̂ dentro da amostra no treino de cada dobra; π̂ fora da amostra no I de Moran. Os resíduos atuais são logit; aqui padronizamos a versão na escala da proporção.
2. ρ̂, krigagem LOSO e CAR rodam sobre z, com o mesmo `CORTE`, a mesma matriz W, as mesmas dobras e sementes.
3. A previsão volta à escala da proporção com sd: π̃ᵢ = π̂ᵢ + ẑᵢ·sdᵢ (limitado a (0, 1)). Métricas: as mesmas das células acima.
4. I de Moran fora da amostra sobre z = (y/n − π)/sd, com π e θ fora da amostra de cada dobra (π̃ para os modelos corrigidos).
5. Critérios: os mesmos de antes (queda do I de Moran >= 0,05 e MAE por UF LOSO <= `limite`). Se a versão ponderada mudasse alguma decisão, isso é registrado como limitação; a decisão não muda.

In [21]:
def _pond_fold(args):
    tr, te = args
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        a = CK.ajustar(df.iloc[tr], mapa)
        pi_tr = np.asarray(CK.prever(a, df.iloc[tr], mapa), float)
        z_tr, _ = S.residuo_pearson(y[tr], n[tr], pi_tr, E.prever_theta(a, df.iloc[tr], mapa))
        pi_te = np.asarray(CK.prever(a, df.iloc[te], mapa), float)
        th_te = E.prever_theta(a, df.iloc[te], mapa)
        sd_te = S.sd_betabinomial(pi_te, n[te], th_te)
        rho, k = S.krigagem_residuos(z_tr, lat, lon, tr, te, uf, CORTE, sd_teste=sd_te)
        par, u = S.car_residuos(z_tr, W, tr, te, sd_teste=sd_te)
    lim = lambda p: np.clip(p, 1e-6, 1 - 1e-6)
    return dict(rho=rho, lam=par["lam"], pi=pi_te, th=th_te, krig=lim(pi_te + k), car=lim(pi_te + u),
                avisos=sorted({str(w.message)[:90] for w in avisos}))
with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
    PND = list(ex.map(_pond_fold, foldsL))
PI_P, TH_P = np.full(len(df), np.nan), np.full(len(df), np.nan)
PIK_P, PIC_P = np.full(len(df), np.nan), np.full(len(df), np.nan)
for (tr, te), r_ in zip(foldsL, PND):
    PI_P[te], TH_P[te], PIK_P[te], PIC_P[te] = r_["pi"], r_["th"], r_["krig"], r_["car"]
print("diferença máx. entre o π fora da amostra desta célula e o do cache LOSO:", float(np.nanmax(np.abs(PI_P - PIL[NOME_F]))))
print("avisos nas dobras:", sorted({a for r_ in PND for a in r_["avisos"]}) or "nenhum")
def moran_z(pi): return S.moran_residuos(S.residuo_pearson(y, n, pi, TH_P)[0], W, permutacoes=999, semente=0)["I"]
I_base_p, I_k_p, I_c_p = moran_z(PIL[NOME_F]), moran_z(PIK_P), moran_z(PIC_P)
EK, EC, EB = erros(PIK_P, "LOSO", foldsL), erros(PIC_P, "LOSO", foldsL), erros(PIL[NOME_F], "LOSO", foldsL)
rho_p = np.array([r_["rho"] for r_ in PND]); lam_p = np.array([r_["lam"] for r_ in PND])
linhas = []
def _l(modelo, metrica, atual, pond, ca=None, cp=None):
    linhas.append(dict(modelo=modelo, metrica=metrica, atual=atual, ponderado=pond, criterio_atual=ca, criterio_ponderado=cp))
for nome, E_, I_p, I_a, mae_a, reg_a in [("krig", EK, I_k_p, I_depois, MAE_K, float(kl.loc[NOME_F + "_krig", "mae_reg_pp"])),
                                         ("car", EC, I_c_p, IC_depois, MAE_C, float(cl.loc[NOME_F + "_car", "mae_reg_pp"]))]:
    if nome == "krig": _l(nome, "rho_media", float(kl.loc[NOME_F + "_krig", "rho_media"]), float(rho_p.mean()))
    else: _l(nome, "lam_media", float(cl.loc[NOME_F + "_car", "lam_media"]), float(lam_p.mean()))
    _l(nome, "mae_uf_pp", mae_a, E_["mae_uf_pp"], bool(mae_a <= LIM), bool(E_["mae_uf_pp"] <= LIM))
    _l(nome, "mae_reg_pp", reg_a, E_["mae_reg_pp"])
    _l(nome, "moran_oos_base", I_antes, I_base_p)
    _l(nome, "moran_oos_modelo", I_a, I_p)
    _l(nome, "queda_moran", I_antes - I_a, I_base_p - I_p, bool(I_antes - I_a >= 0.05), bool(I_base_p - I_p >= 0.05))
SENS = pd.DataFrame(linhas); SENS.to_csv(TAB / "sensibilidade_ponderada_krig_car.csv", index=False)
print(SENS.to_string(index=False))
print("decisão (atual -> ponderada):", {m: (bool(g.criterio_atual.dropna().all()), bool(g.criterio_ponderado.dropna().all())) for m, g in SENS.groupby("modelo")})

diferença máx. entre o π fora da amostra desta célula e o do cache LOSO: 0.0
avisos nas dobras: ['df_resid differs from nobs - k_params']


modelo          metrica  atual  ponderado criterio_atual criterio_ponderado
  krig        rho_media 0.0050     0.0075           None               None
  krig        mae_uf_pp 1.0771     1.0788           True               True
  krig       mae_reg_pp 0.4283     0.6239           None               None
  krig   moran_oos_base 0.4224     0.3728           None               None
  krig moran_oos_modelo 0.4222     0.3818           None               None
  krig      queda_moran 0.0003    -0.0091          False              False
   car        lam_media 0.7981     0.8959           None               None
   car        mae_uf_pp 1.0037     0.9894           True               True
   car       mae_reg_pp 0.5634     0.6074           None               None
   car   moran_oos_base 0.4224     0.3728           None               None
   car moran_oos_modelo 0.3949     0.3476           None               None
   car      queda_moran 0.0275     0.0251          False              False
decisão (atu

**Leitura (sensibilidade informativa; as decisões não mudam).** Com resíduos de Pearson, nenhum critério muda de lado. Krigagem: ρ̂ médio 0,0075 (era 0,0050), MAE por UF LOSO 1,0788 p.p. (era 1,0771; limite 1,2515, passa nos dois), e a queda do I de Moran é −0,0091 (era +0,0003), longe dos 0,05 exigidos: rejeitada nas duas versões. CAR: MAE por UF 0,9894 p.p. (era 1,0037, passa nos dois), MAE regional 0,607 (era 0,563) e queda do I de 0,0251 (era 0,0275), abaixo dos 0,05: rejeitado nas duas. O I de Moran fora da amostra do modelo base cai de 0,4224 para 0,3728 só por padronizar, o que mostra que parte da correlação espacial vem dos municípios pequenos. Os números atuais das células acima ficaram idênticos na reexecução. Tabela: `results/tables/sensibilidade_ponderada_krig_car.csv`.

*Números da execução de 2026-11-22.*